In [3]:
import numpy as np
import tidy3d as td
import tidy3d.web as web
import autograd as ad
from autograd import value_and_grad
import autograd.numpy as anp
from tidy3d.plugins.autograd import rescale
import matplotlib.pyplot as plt
import pyvista as pv
import time
lambda_0 = 1.05
f0 = td.C_0 / lambda_0
fwidth = f0 / 100
#run_time = 25 / fwidth
run_time = 15 / fwidth

hx = hy = hz = 0.6
lz = 2
r_design = 0.3
r_0 = 0.08
width = 0.003
#dl = 40
nx = ny = 300
nz = 300
eps_spheres = 3.5**2
eps_embedding = 1.5**2

R = 0.2
zmax = 0.0
shift = 0.0
shift2 = 0.0
safespace = 5.0
num = 5
ni = np.linspace(0, 360.0, num + 1)[:-1]
x = R * np.cos(ni * np.pi / 180.0)
y = R * np.sin(ni * np.pi / 180.0)
z = np.linspace(-zmax, zmax, x.shape[0])
positions = np.array([x, y, z]).T
radii = np.ones_like(positions[:, 0]) * r_0
params_0 = np.column_stack((radii, positions))


dl = 0.005

b_spec = td.BoundarySpec(
    x=td.Boundary(minus=td.Periodic(), plus=td.Periodic()),
    y=td.Boundary(minus=td.Periodic(), plus=td.Periodic()),
    z=td.Boundary(minus=td.PML(), plus=td.PML()),
)

cur_it = 0


def make_sphere_structures(params):
    sphere_medium = td.Medium(permittivity=eps_spheres)
    structures = []

    for r, cx, cy, cz in params:
        structures.append(
            td.Structure(
                geometry=td.Sphere(center=(cx, cy, cz), radius=r),
                medium=sphere_medium,
            )
        )
    return structures

def plot_spheres(params, convert_from_index_units=True):
    """
    params shape: (N,4) with columns [radius, cx, cy, cz]
    """
    p = pv.Plotter()
    params_plot = params

    for r, cx, cy, cz in params_plot:
        sphere = pv.Sphere(
            radius=float(r),
            center=(float(cx), float(cy), float(cz)),
            theta_resolution=48,
            phi_resolution=48,
        )
        p.add_mesh(sphere, opacity=0.7, show_edges=False)

    p.add_axes()
    p.show_grid()
    p.show()
 

def make_sim(params):
    structures = make_sphere_structures(params)

    # grid_spec = td.GridSpec.auto(
    #     wavelength=lambda_0,
    #     min_steps_per_wvl=60,
    # )
    
    # grid_spec = td.GridSpec.uniform(dl=0.005)
 
    grid_spec = td.GridSpec.auto(
        wavelength=lambda_0,
        min_steps_per_wvl=30,   # or 40
        override_structures=[
            td.MeshOverrideStructure(
                geometry=td.Box(center=(0, 0, 0), size=(0.5, 0.5, 0.5)),
                dl=(0.004, 0.004, 0.004),
            )
        ],
    )

    sim_x_pol = td.Simulation(
        size=(hx, hy, lz),
        grid_spec=grid_spec,
        medium=td.Medium(permittivity=eps_embedding),
        structures=structures,
        sources=[source_x_pol],
        monitors=[flux_monitor, perm_monitor, field_monitor],
        run_time=run_time,
        shutoff=1e-4,
        boundary_spec=b_spec,
    )

    sim_y_pol = td.Simulation(
        size=(hx, hy, lz),
        grid_spec=grid_spec,
        medium=td.Medium(permittivity=eps_embedding),
        structures=structures,
        sources=[source_y_pol],
        monitors=[flux_monitor, perm_monitor, field_monitor],
        run_time=run_time,
        shutoff=1e-4,
        boundary_spec=b_spec,
    )

    return sim_x_pol, sim_y_pol

def measure_transmittance(sim_data):
    return -sim_data["T"].flux.values

def measure_reflectance(sim_data):
    return -sim_data["R"].flux.values

def obj(params):
    #sim_x_pol, sim_y_pol = make_sim(params, grad)
    sim_x_pol, sim_y_pol = make_sim(params)
    sim_data_x_pol = web.run(sim_x_pol, task_name=f"x_pol_{cur_it}", folder_name="Five_Spheres", verbose=False)
    sim_data_y_pol = web.run(sim_y_pol, task_name=f"y_pol_{cur_it}", folder_name="Five_Spheres", verbose=False)

    refl_x_pol = measure_reflectance(sim_data_x_pol)
    #return refl_x_pol
    refl_y_pol = measure_reflectance(sim_data_y_pol)
    print("refl x", refl_x_pol, "refl y", refl_y_pol)
    return anp.abs(refl_x_pol - refl_y_pol)

maxeval = 1

gaussian = td.GaussianPulse(freq0=f0, fwidth=fwidth)

z_pos_source = -hz/2 - (lz/2-hz/2)/2
z_pos_flux = -lz / 2 + 0.2

source_x_pol = td.PlaneWave(
    source_time=gaussian,
    size=(td.inf, td.inf, 0),
    center=(0, 0, z_pos_source),
    direction='+',
    pol_angle=0
)
source_y_pol = td.PlaneWave(
    source_time=gaussian,
    size=(td.inf, td.inf, 0),
    center=(0, 0, z_pos_source),
    direction='+',
    pol_angle=anp.pi / 2
)

flux_monitor = td.FieldMonitor(
    size=(td.inf, td.inf, 0),
    center=(0, 0, z_pos_flux),
    freqs=[f0],
    name="R",
)

perm_monitor = td.PermittivityMonitor(
    center=(0, 0, 0),
    size=(0, 2*r_design + 0.3, 2*r_design),
    freqs=[f0],
    name='Permittivity Monitor'
)

field_monitor = td.FieldMonitor(
    center=(0, 0, 0),
    size=(0, 2*r_design + 0.3, 2*r_design),
    freqs=[f0],
    name='Field Monitor'
)

In [4]:
def time_forward_only(params):
    t0 = time.perf_counter()
    v = obj(params)               
    t1 = time.perf_counter()
    return float(np.ravel(v)[0]), (t1 - t0)

v, tdiff = time_forward_only(params_0)
print("time forward", tdiff)

refl x [0.03427357] refl y [0.03649177]
time forward 148.23179011931643


In [5]:
import h5py
dl = 0.005
with h5py.File(f"data_primitive_rcd_init_R_{R}_rinit_{r_0}_lz_{lz}_dlauto_{dl}_weight_{width}_dist_flux_{z_pos_flux}_source_{z_pos_source}.h5",'w') as f:

    f["value"] = v
    f["time_forward"] =tdiff